# 🤖 Agentic Drilling Advisor
Wiper trip GO/NO-GO advisory for well 16A(78)-32.

**Prerequisites:** Run `01_knowledge_base.ipynb` first.

**LLM:** Qwen 2.5 7B via Ollama — `ollama pull qwen2.5:7b`

## 1. Setup

In [ ]:
import os, re, warnings
import numpy as np, pandas as pd
warnings.filterwarnings('ignore')
MODEL_NAME = 'qwen2.5:7b'

## 2. Load Data

In [ ]:
COL_MAP = {
    'Time':'Time','Weight on Bit':'WOB','ROP Depth/Hour':'ROP',
    'Top Drive RPM':'RPM','Top Drive Torque (ft-lbs)':'TRQ',
    'Flow In':'FLOW_IN','Pump Pressure':'SPP','Depth Hole TVD':'DEPTH',
    'Differential Pressure':'DIFF_P','Downhole Torque':'DH_TRQ',
    'Hookload':'HOOKLOAD','Gas Total - units':'GAS',
    'Return Flow':'RETURN_FLOW','Pit G/L Active':'PIT_GL',
    'Block Position':'BLOCK_POS','MWD Inclination':'MWD_INC',
    'On Bottom':'ON_BOTTOM','MUD TEMP':'MUD_TEMP',
    'Trip G/L':'TRIP_GL','Trip Volume Active':'TRIP_VOL',
}
UNITS = {'WOB':'klbs','ROP':'m/hr','RPM':'rpm','TRQ':'ft-lbs',
         'FLOW_IN':'gpm','SPP':'psi','HOOKLOAD':'klbs','DEPTH':'m'}
KEY_PARAMS = ['WOB','ROP','RPM','TRQ','SPP','FLOW_IN','HOOKLOAD','DH_TRQ','DIFF_P']

csv_path = os.path.join(os.getcwd(), '16A(78)-32_time_data_10s_intervals.csv')
df = pd.read_csv(csv_path)
df.rename(columns=COL_MAP, inplace=True)
df = df.iloc[::10].reset_index(drop=True)
df['Time'] = pd.to_datetime(df['Time'], format='mixed', dayfirst=False)
df[df.select_dtypes('number').columns] = df.select_dtypes('number').replace(-999.25, np.nan)
scols = [c for c in KEY_PARAMS+['BLOCK_POS','GAS','RETURN_FLOW','PIT_GL','MWD_INC','MUD_TEMP'] if c in df.columns]
df[scols] = df[scols].ffill().fillna(0)
print(f'Sensor data: {len(df):,} rows, {df["Time"].min()} to {df["Time"].max()}')

import chromadb
from chromadb.config import Settings
from chromadb.utils.embedding_functions import SentenceTransformerEmbeddingFunction

chroma_dir = os.path.join(os.getcwd(), 'chroma_db')
client = chromadb.PersistentClient(path=chroma_dir, settings=Settings(anonymized_telemetry=False))
emb_fn = SentenceTransformerEmbeddingFunction(model_name='all-MiniLM-L6-v2')
collection = client.get_or_create_collection('daily_reports', embedding_function=emb_fn)
print(f'Knowledge base: {collection.count()} chunks loaded')

## 3. Define Agent Tools

In [ ]:
from langchain_core.tools import tool

@tool
def wiper_trip_assessment() -> str:
    """Perform a comprehensive wiper trip risk assessment by analyzing ALL current
    sensor parameters against known risk thresholds. Returns a risk score (0-100),
    individual parameter flags, and a GO/NO-GO recommendation. Always call this
    tool first when asked about wiper trip decisions."""
    flags = []
    risk_score = 0
    n = min(100, len(df))
    window = df.iloc[-n:]
    latest = df.iloc[-1]

    trq_start, trq_end = window['TRQ'].iloc[0], window['TRQ'].iloc[-1]
    trq_pct = ((trq_end - trq_start) / abs(trq_start) * 100) if trq_start else 0
    if trq_pct > 10:
        flags.append(f'WARNING TORQUE rising {trq_pct:+.1f}% (current: {trq_end:.0f} ft-lbs)')
        risk_score += 25
    elif trq_pct > 5:
        flags.append(f'CAUTION TORQUE moderately up {trq_pct:+.1f}%')
        risk_score += 10
    else:
        flags.append(f'OK TORQUE stable ({trq_pct:+.1f}%)')

    hl_start, hl_end = window['HOOKLOAD'].iloc[0], window['HOOKLOAD'].iloc[-1]
    hl_pct = ((hl_end - hl_start) / abs(hl_start) * 100) if hl_start else 0
    hl_std = window['HOOKLOAD'].std()
    if abs(hl_pct) > 10 or hl_std > df['HOOKLOAD'].std() * 1.5:
        flags.append(f'WARNING HOOKLOAD erratic: {hl_pct:+.1f}% change, std={hl_std:.1f}')
        risk_score += 20
    elif abs(hl_pct) > 5:
        flags.append(f'CAUTION HOOKLOAD shifting {hl_pct:+.1f}%')
        risk_score += 8
    else:
        flags.append(f'OK HOOKLOAD stable ({hl_pct:+.1f}%)')

    spp_start, spp_end = window['SPP'].iloc[0], window['SPP'].iloc[-1]
    spp_pct = ((spp_end - spp_start) / abs(spp_start) * 100) if spp_start else 0
    if spp_pct > 10:
        flags.append(f'WARNING SPP rising {spp_pct:+.1f}% (possible pack-off)')
        risk_score += 20
    elif spp_pct > 5:
        flags.append(f'CAUTION SPP moderately up {spp_pct:+.1f}%')
        risk_score += 8
    else:
        flags.append(f'OK SPP stable ({spp_pct:+.1f}%)')

    rop_start, rop_end = window['ROP'].iloc[0], window['ROP'].iloc[-1]
    rop_pct = ((rop_end - rop_start) / abs(rop_start) * 100) if rop_start else 0
    if rop_pct < -15:
        flags.append(f'WARNING ROP dropping {rop_pct:+.1f}% (tight hole indicator)')
        risk_score += 15
    elif rop_pct < -5:
        flags.append(f'CAUTION ROP declining {rop_pct:+.1f}%')
        risk_score += 5
    else:
        flags.append(f'OK ROP normal ({rop_pct:+.1f}%)')

    dp_max = window['DIFF_P'].max()
    dp_thresh = df['DIFF_P'].mean() + 2 * df['DIFF_P'].std()
    if dp_max > dp_thresh:
        flags.append(f'WARNING DIFF_P spikes (max={dp_max:.0f}, threshold={dp_thresh:.0f})')
        risk_score += 10
    else:
        flags.append(f'OK DIFF_P within normal range')

    if 'GAS' in df.columns:
        gas_cur = latest.get('GAS', 0)
        gas_mean = df['GAS'].mean()
        if gas_cur > gas_mean * 2 and gas_cur > 0:
            flags.append(f'WARNING GAS elevated: {gas_cur:.1f} vs avg {gas_mean:.1f}')
            risk_score += 10
        else:
            flags.append(f'OK GAS normal ({gas_cur:.1f})')

    inc = latest.get('MWD_INC', 0) if 'MWD_INC' in df.columns else 0
    if inc > 30:
        flags.append(f'CAUTION HIGH ANGLE: {inc:.1f} deg (cuttings settling risk)')
        risk_score += 10

    risk_score = min(risk_score, 100)
    if risk_score >= 60:
        decision = 'RECOMMEND WIPER TRIP - Multiple high-risk indicators'
    elif risk_score >= 35:
        decision = 'CONSIDER WIPER TRIP - Elevated risk, monitor closely'
    else:
        decision = 'NO WIPER TRIP NEEDED - Conditions within normal range'

    depth = latest.get('DEPTH', 0)
    lines = [
        f'WIPER TRIP RISK ASSESSMENT',
        f'Well: FORGE 16A(78)-32',
        f'Time: {latest["Time"]}',
        f'Depth: {depth:,.0f} m TVD | Inclination: {inc:.1f} deg',
        f'Risk Score: {risk_score}/100',
        f'',
        f'Parameter Flags:',
    ]
    lines.extend(flags)
    lines.append('')
    lines.append('Decision: ' + decision)
    lines.append('')
    lines.append('Current Readings:')
    for p in KEY_PARAMS:
        if p in df.columns:
            lines.append(f'  {p}: {latest[p]:.2f} {UNITS.get(p, "")}')
    return '\n'.join(lines)


@tool
def report_search(query: str) -> str:
    """Search 163 historical daily drilling reports for well 16A(78)-32.
    Reports cover wiper trips, reaming, tight spots, high torque, stuck pipe,
    pack-off, drag events, and crew operational decisions."""
    try:
        results = collection.query(query_texts=[query], n_results=5)
        if not results or not results['documents'][0]:
            return 'No relevant reports found.'
        lines = []
        for i, (doc, meta) in enumerate(zip(results['documents'][0], results['metadatas'][0])):
            date = meta.get('date_display', meta.get('date', 'Unknown'))
            depth = meta.get('depth_md', 'N/A')
            events = meta.get('event_types', 'none')
            lines.append(f'Report {i+1}:')
            lines.append(f'Date: {date} | Depth: {depth} | Events: {events}')
            lines.append(f'{doc[:500]}\n')
        return '\n'.join(lines)
    except Exception as e:
        return f'Error: {e}'


@tool
def sensor_analysis(query: str) -> str:
    """Analyze a specific sensor parameter in detail. Available: WOB, ROP, RPM,
    TRQ, SPP, FLOW_IN, HOOKLOAD, DH_TRQ, DIFF_P, GAS, DEPTH."""
    q = query.lower()
    try:
        param = 'TRQ'
        aliases = {'torque':'TRQ','pressure':'SPP','weight':'WOB','rop':'ROP',
                   'hookload':'HOOKLOAD','flow':'FLOW_IN','gas':'GAS','depth':'DEPTH','rpm':'RPM'}
        for a, p in aliases.items():
            if a in q: param = p; break
        for p in KEY_PARAMS:
            if p.lower() in q: param = p; break
        n = min(100, len(df))
        window = df.iloc[-n:]
        cur = window[param].iloc[-1]
        start_val = window[param].iloc[0]
        pct = ((cur-start_val)/abs(start_val)*100) if start_val else 0
        trend = 'INCREASING' if pct>2 else 'DECREASING' if pct<-2 else 'STABLE'
        mean, std = df[param].mean(), df[param].std()
        anom_count = len(df[df[param] > mean + 2.5*std])
        return (f'{param} Analysis:\n'
                f'  Current: {cur:.2f} {UNITS.get(param,"")}\n'
                f'  Trend: {trend} ({pct:+.1f}% over last {n} points)\n'
                f'  Mean: {mean:.2f}, Std: {std:.2f}\n'
                f'  Range: {df[param].min():.2f} to {df[param].max():.2f}\n'
                f'  Anomalies (>2.5 sigma): {anom_count} out of {len(df):,} points')
    except Exception as e:
        return f'Error: {e}'


@tool
def well_context() -> str:
    """Get well identification, depth, trajectory, and data coverage."""
    row = df.iloc[-1]
    inc = row.get('MWD_INC', 0) if 'MWD_INC' in df.columns else 0
    return (f'Well: FORGE 16A(78)-32 (Utah FORGE geothermal, Milford UT)\n'
            f'Depth: {row.get("DEPTH",0):,.0f} m TVD | Inclination: {inc:.1f} deg\n'
            f'Data: {len(df):,} points, {df["Time"].min()} to {df["Time"].max()}\n'
            f'Reports: 163 daily drilling PDFs, 125+ mined events\n'
            f'Bit: 8.5 in | Sensors: 36 channels @ 10-sec intervals')

ALL_TOOLS = [wiper_trip_assessment, report_search, sensor_analysis, well_context]
print(f'Defined {len(ALL_TOOLS)} tools: {[t.name for t in ALL_TOOLS]}')

## 4. Create Agent (Qwen 2.5 via Ollama)

In [ ]:
from langchain_ollama import ChatOllama
from langchain_core.messages import HumanMessage, SystemMessage, AIMessage
from langgraph.prebuilt import create_react_agent
from langgraph.checkpoint.memory import MemorySaver

SYSTEM_PROMPT = """You are an expert drilling engineering advisor for well FORGE 16A(78)-32.
Your PRIMARY job is to advise whether a wiper trip should be performed.

WORKFLOW - Always follow this order:
1. Call wiper_trip_assessment FIRST to get the current sensor risk score
2. Call report_search to find historical precedents matching current conditions
3. Correlate the sensor risk score with what happened historically

OUTPUT FORMAT:
### CURRENT SENSOR STATUS
Summarize the risk score and key flags from wiper_trip_assessment

### HISTORICAL PRECEDENT
What the reports show - cite specific dates, depths, and outcomes

### RECOMMENDATION
Clear GO / NO-GO decision with confidence level (HIGH/MEDIUM/LOW)
Specific actions: trip parameters, depth intervals, monitoring points

### MONITORING PLAN
What to watch if continuing drilling without a wiper trip

Be specific. Cite actual values and dates. Safety first."""

llm = ChatOllama(model=MODEL_NAME, temperature=0)
memory = MemorySaver()
agent = create_react_agent(
    model=llm, tools=ALL_TOOLS,
    prompt=SystemMessage(content=SYSTEM_PROMPT),
    checkpointer=memory,
)
print(f'Agent ready: {MODEL_NAME}')

## 5. Wiper Trip Advisory

In [ ]:
from IPython.display import display, Markdown

def ask(question, thread='default'):
    """Send a question to the drilling advisor and display as rendered Markdown."""
    config = {'configurable': {'thread_id': thread}}
    display(Markdown(f'**Question:** {question}'))
    display(Markdown('*Thinking...*'))
    result = agent.invoke({'messages': [HumanMessage(content=question)]}, config=config)
    tools_used = []
    response = ''
    for msg in result['messages']:
        if isinstance(msg, AIMessage):
            if msg.content: response = msg.content
            if hasattr(msg, 'tool_calls') and msg.tool_calls:
                for tc in msg.tool_calls:
                    tools_used.append(tc['name'])
    if tools_used:
        display(Markdown(f'**Tools used:** {", ".join(tools_used)}'))
    display(Markdown('---'))
    display(Markdown(response))
    return response

In [ ]:
response = ask(
    'Based on current sensor readings and historical report data, '
    'should we perform a wiper trip right now? '
    'Analyze sensor risk, find similar historical situations, and give a clear GO or NO-GO.'
)

In [ ]:
response = ask('What historical wiper trip events happened at similar depths and conditions?')

In [ ]:
response = ask('Analyze the torque and hookload trends in detail. Are they concerning?')

## 6. Interactive Mode (Optional)
Type `quit` to exit.

In [ ]:
while True:
    q = input('You: ').strip()
    if q.lower() in ('quit','exit','q',''): print('Goodbye!'); break
    response = ask(q)
    print('\n' + '='*60 + '\n')